# Entraînement LoRA SDXL — personnage "Solle" (style comic)

Notebook **Kaggle** (GPU T4 16 Go gratuit) pour entraîner un LoRA SDXL à partir des 35 images préparées.

## Avant de lancer
1. Sur Kaggle : crée un **Dataset** privé contenant le contenu de ton dossier local `training/dataset/`
   (les 35 `.png` + 35 `.txt`). Nomme-le par ex. `solle-dataset`.
2. Dans ce notebook : menu de droite **Add Input** -> ajoute ton dataset `solle-dataset`.
   Il sera monté sous `/kaggle/input/solle-dataset/`.
3. Menu de droite : **Accelerator = GPU T4 x2** (ou P100), **Internet = ON**.
4. Exécute les cellules de haut en bas.

À la fin, tu télécharges `solle_lora.safetensors` (~100-200 Mo) et tu l'utilises en local dans ComfyUI.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer sd-scripts (moteur kohya)

In [ ]:
%cd /kaggle/working
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
# Version figée connue pour fonctionner avec SDXL LoRA
!git checkout v0.8.7
!pip install -q -r requirements.txt
!pip install -q xformers bitsandbytes
# accelerate en mode non-interactif, 1 GPU, fp16
!mkdir -p ~/.cache/huggingface/accelerate
import os, textwrap
cfg = textwrap.dedent('''\
    compute_environment: LOCAL_MACHINE
    distributed_type: 'NO'
    mixed_precision: fp16
    num_processes: 1
    use_cpu: false
''')
open(os.path.expanduser('~/.cache/huggingface/accelerate/default_config.yaml'),'w').write(cfg)
print('OK')

## 3. Télécharger le modèle de base SDXL

In [ ]:
%cd /kaggle/working
!mkdir -p models
# SDXL 1.0 base (~6.9 Go)
!wget -q -O models/sd_xl_base_1.0.safetensors \
  https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors
!ls -lh models/

## 4. Préparer les dossiers de dataset

kohya attend une structure `<dossier>/<repeats>_<nom>/` contenant images + .txt.
Le préfixe `10_` = 10 répétitions par image et par epoch.
Avec 35 images x 10 repeats = 350 steps/epoch ; on vise ~2000 steps au total.

In [ ]:
import os, glob, shutil

SRC = '/kaggle/input/solle-dataset'        # adapte si ton dataset a un autre nom
DST = '/kaggle/working/train_data/10_sollechar'
os.makedirs(DST, exist_ok=True)

n = 0
for f in glob.glob(os.path.join(SRC, '*')):
    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.txt')):
        shutil.copy(f, DST)
        n += 1
print(f'{n} fichiers copiés vers {DST}')
print('Exemple caption :', open(glob.glob(DST + '/*.txt')[0]).read())

## 5. Lancer l'entraînement

~2000 steps, rank 32. Sur T4 compte ~45-90 min. Le LoRA est sauvé toutes les 2 epochs.

In [ ]:
%cd /kaggle/working/sd-scripts
!accelerate launch --num_cpu_threads_per_process 2 sdxl_train_network.py \
  --pretrained_model_name_or_path=/kaggle/working/models/sd_xl_base_1.0.safetensors \
  --train_data_dir=/kaggle/working/train_data \
  --output_dir=/kaggle/working/output \
  --output_name=solle_lora \
  --resolution=1024,1024 \
  --network_module=networks.lora \
  --network_dim=32 \
  --network_alpha=16 \
  --train_batch_size=1 \
  --gradient_checkpointing \
  --gradient_accumulation_steps=1 \
  --max_train_steps=2000 \
  --learning_rate=1e-4 \
  --unet_lr=1e-4 \
  --text_encoder_lr=5e-5 \
  --lr_scheduler=cosine \
  --lr_warmup_steps=100 \
  --optimizer_type=AdamW8bit \
  --mixed_precision=fp16 \
  --save_precision=fp16 \
  --cache_latents \
  --save_model_as=safetensors \
  --save_every_n_epochs=2 \
  --max_data_loader_n_workers=2 \
  --seed=42 \
  --xformers \
  --no_half_vae

## 6. Récupérer le LoRA

Les fichiers sont dans `/kaggle/working/output/`. Ouvre l'onglet **Output** (panneau de droite)
pour télécharger `solle_lora.safetensors`. Garde aussi une version intermédiaire
(`solle_lora-000004.safetensors` etc.) : la dernière epoch sur-apprend parfois.

In [ ]:
!ls -lh /kaggle/working/output/